In [7]:
import polars as pl
import pandas as pd

yyyymm         = '202605'
reporting_date = '2026-05'
usd2thb = 33

# Prep Data

In [2]:
loandata = pl.read_excel(
  '../data/BB202605- XIGNAL Jun26 add ISIC.xlsx', 
  sheet_name = '#LN00157'
)
loandata.columns = [i.lower() for i in loandata.columns]


ef_asset = pl.read_excel(
  '../data/BB202605- XIGNAL Jun26 add ISIC.xlsx', 
  sheet_name = 'Asset'
)
ef_asset.columns = [i.lower() for i in ef_asset.columns]

ef_revenue = pl.read_excel(
  '../data/BB202605- XIGNAL Jun26 add ISIC.xlsx', 
  sheet_name = 'Revenue'
)
ef_revenue.columns = [i.lower() for i in ef_revenue.columns]

tbl_isic = pl.read_excel(
  '../data/BB202605- XIGNAL Jun26 add ISIC.xlsx', 
  sheet_name = 'tbl_ISIC'
)
tbl_isic.columns = [i.lower() for i in tbl_isic.columns]


mapping_isic_clean = (
  pl.read_excel('../data/mapping_isic_clean.xlsx')
  .with_columns(
    clean_iccode  = pl.col('clean_iccode').cast(pl.Utf8).str.zfill(4),
    clean_iccode2 = pl.col('clean_iccode2').cast(pl.Utf8).str.zfill(4),
  )
)

Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 13, falling back to string


In [3]:
ef_revenue_isic = (
  tbl_isic
  .join(
    ef_revenue,
    how      = 'left',
    left_on  = 'bea code',
    right_on = 'classification_code',
  )
  .group_by('clean_iccode', 'ic sector')
  .agg(
    scope_1_r = (pl.col('scope_1')*pl.col('weight')).sum(),
    scope_2_r = (pl.col('scope_2')*pl.col('weight')).sum(),
    scope_3_r = (pl.col('scope_3')*pl.col('weight')).sum()
  )
  .sort('clean_iccode')
)

ef_asset_isic = (
  tbl_isic
  .join(
    ef_asset,
    how      = 'left',
    left_on  = 'bea code',
    right_on = 'classification_code',
  )
  .group_by('clean_iccode', 'ic sector')
  .agg(
    scope_1_a = (pl.col('scope_1')*pl.col('weight')).sum(),
    scope_2_a = (pl.col('scope_2')*pl.col('weight')).sum(),
    scope_3_a = (pl.col('scope_3')*pl.col('weight')).sum()
  )
  .sort('clean_iccode')
)

# EDA

In [4]:
(
  loandata
  .group_by('cus_type2')
  .agg(
    n   = pl.len(),
    amt = pl.col('loanamt').sum()
  )
  .sort('cus_type2')
  .with_columns(
    pct_n   = pl.col('n')   / pl.col('n').sum(),
    pct_amt = pl.col('amt') / pl.col('amt').sum(),
  )
)

cus_type2,n,amt,pct_n,pct_amt
str,u32,f64,f64,f64
"""Corporate""",5047,9.4789e9,0.185361,0.237418
"""Individual""",22181,3.0446e10,0.814639,0.762582


In [5]:
(
  loandata
  .group_by('grpbusi')
  .agg(
    n   = pl.len(),
    amt = pl.col('loanamt').sum()
  )
  .sort('grpbusi')
  .with_columns(
    pct_n   = pl.col('n')   / pl.col('n').sum(),
    pct_amt = pl.col('amt') / pl.col('amt').sum(),
  )
)

grpbusi,n,amt,pct_n,pct_amt
str,u32,f64,f64,f64
"""Agriculture""",499,6.0684e8,0.018327,0.0152
"""Chemical & Phamaceuticals""",78,1.0601e8,0.002865,0.002655
"""Commercial""",14362,2.0309e10,0.527472,0.50867
"""Construction & Real Estate""",2688,4.8158e9,0.098722,0.120621
"""Education""",150,2.5419e8,0.005509,0.006367
…,…,…,…,…
"""Individual""",39,1.9403e7,0.001432,0.000486
"""Industrial""",3142,4.8430e9,0.115396,0.121301
"""Mining""",22,4.3075e7,0.000808,0.001079


# FE cal

In [6]:
loandata_fe = (
  loandata
  .with_columns(
    total_income     = pl.col('total_income').cast(pl.Float64, strict = False),
    debt_plus_equity = pl.col('debt_plus_equity').cast(pl.Float64, strict = False),
    clean_iccode     = pl.col('bot_isic_4_0').str.zfill(7).str.slice(1, 4)
  )
  .with_columns(
    clean_iccode = (
      pl.when(pl.col('grpbusi') == 'Individual')
        .then(pl.lit('9609'))
        .otherwise(pl.col('clean_iccode'))
    )
  )
  .join(
    mapping_isic_clean,
    how      = 'left',
    left_on  = 'clean_iccode',
    right_on = 'clean_iccode',
  )
  .with_columns(
    clean_iccode2 = pl.col('clean_iccode2').fill_null(pl.col('clean_iccode'))
  )
  .join(
    ef_revenue_isic.drop('ic sector'),
    how     = 'left', 
    left_on  = 'clean_iccode2',
    right_on = 'clean_iccode',
  )
  .join(
    ef_asset_isic.drop('ic sector'),
    how     = 'left', 
    left_on  = 'clean_iccode2',
    right_on = 'clean_iccode',
  )
  .with_columns(
    attribution_factor = (
      pl.when(
          pl.col('debt_plus_equity') > 0,
          pl.col('debt_plus_equity') < pl.col('loanamt')
        )
        .then(1)
        .when(
          pl.col('debt_plus_equity') > 0,
        )
        .then(pl.col('loanamt') / pl.col('debt_plus_equity'))
        .otherwise(1)
    ),
    score = (
      pl.when(pl.col('debt_plus_equity') > 0)
        .then(4)
        .otherwise(5)
    )
  )
  .with_columns(
    fe_scope_1 = (
      pl.when(pl.col('score') == 4)
        .then(pl.col('attribution_factor') * pl.col('total_income') * pl.col('scope_1_r') / usd2thb)
        .otherwise(pl.col('loanamt') * pl.col('scope_1_a') / usd2thb)
    ),
    fe_scope_2 = (
      pl.when(pl.col('score') == 4)
        .then(pl.col('attribution_factor') * pl.col('total_income') * pl.col('scope_2_r') / usd2thb)
        .otherwise(pl.col('loanamt') * pl.col('scope_2_a') / usd2thb)
    ),
    fe_scope_3 = (
      pl.when(pl.col('score') == 4)
        .then(pl.col('attribution_factor') * pl.col('total_income') * pl.col('scope_3_r') / usd2thb)
        .otherwise(pl.col('loanamt') * pl.col('scope_3_a') / usd2thb)
    ),
  )
)

# Analysis

In [9]:
tbl_port = (
  loandata_fe
  .with_columns(
    weight_amt = pl.col('loanamt') / pl.col('loanamt').sum()
  )
  .with_columns(
    score_weight = pl.col('score') * pl.col('weight_amt')
  )
  .select('loanamt', 'fe_scope_1', 'fe_scope_2', 'fe_scope_3', 'score_weight')
  .sum()
  .with_columns(
    fe_scope_12  = pl.col('fe_scope_1') + pl.col('fe_scope_2'),
    fe_scope_123 = pl.col('fe_scope_1') + pl.col('fe_scope_2') + pl.col('fe_scope_3'),
  )
  .with_columns(
    intensity = pl.col('fe_scope_12') * 1e3 / (pl.col('loanamt') / usd2thb)
  )
  # .write_clipboard()
)

In [ ]:
tbl_industry = (
  loandata_fe
  .with_columns(
    weight_amt = pl.col('loanamt') / pl.col('loanamt').sum().over('grpbusi')
  )
  .with_columns(
    score_weight = pl.col('score') * pl.col('weight_amt')
  )
  .group_by('grpbusi')
  .agg(
    pl.col('loanamt').sum()/1e6,
    pl.col('fe_scope_1').sum()/1e3,
    pl.col('fe_scope_2').sum()/1e3,
    pl.col('fe_scope_3').sum()/1e3,
    pl.col('score_weight').sum()
  )
  .with_columns(
    fe_scope_12  = pl.col('fe_scope_1') + pl.col('fe_scope_2'),
    fe_scope_123 = pl.col('fe_scope_1') + pl.col('fe_scope_2') + pl.col('fe_scope_3'),
  )
  .with_columns(
    intensity = pl.col('fe_scope_12') / (pl.col('loanamt') / usd2thb),
    pct_amt = pl.col('loanamt') / pl.col('loanamt').sum()
  )
  .sort('loanamt', descending=True)
  .write_clipboard()
)

In [ ]:
tbl_by_customer = (
  loandata_fe
  .with_columns(
    weight_amt = pl.col('loanamt') / pl.col('loanamt').sum().over('cus_type2')
  )
  .with_columns(
    score_weight = pl.col('score') * pl.col('weight_amt')
  )
  .group_by('cus_type2')
  .agg(
    pl.col('loanamt').sum()/1e6,
    pl.col('fe_scope_1').sum()/1e3,
    pl.col('fe_scope_2').sum()/1e3,
    pl.col('fe_scope_3').sum()/1e3,
    pl.col('score_weight').sum()
  )
  .with_columns(
    fe_scope_12  = pl.col('fe_scope_1') + pl.col('fe_scope_2'),
    fe_scope_123 = pl.col('fe_scope_1') + pl.col('fe_scope_2') + pl.col('fe_scope_3'),
  )
  .with_columns(
    intensity = pl.col('fe_scope_12') / (pl.col('loanamt') / usd2thb),
    pct_amt = pl.col('loanamt') / pl.col('loanamt').sum()
  )
  # .sort('loanamt', descending=True)
  .sort('cus_type2', 'loanamt', descending=True)
  .write_clipboard()
)

In [10]:
# (
#   loandata_fe
#   .with_columns(
#     weight_amt = pl.col('loanamt') / pl.col('loanamt').sum().over('grpbusi', 'cus_type2')
#   )
#   .with_columns(
#     score_weight = pl.col('score') * pl.col('weight_amt')
#   )
#   .group_by('cus_type2', 'grpbusi')
#   .agg(
#     pl.col('loanamt').sum()/1e6,
#     pl.col('fe_scope_1').sum()/1e3,
#     pl.col('fe_scope_2').sum()/1e3,
#     pl.col('fe_scope_3').sum()/1e3,
#     pl.col('score_weight').sum()
#   )
#   .with_columns(
#     fe_scope_12  = pl.col('fe_scope_1') + pl.col('fe_scope_2'),
#     fe_scope_123 = pl.col('fe_scope_1') + pl.col('fe_scope_2') + pl.col('fe_scope_3'),
#   )
#   .with_columns(
#     intensity = pl.col('fe_scope_12') / (pl.col('loanamt') / usd2thb),
#     pct_amt = pl.col('loanamt') / pl.col('loanamt').sum()
#   )
#   # .sort('loanamt', descending=True)
#   .sort('cus_type2', 'loanamt', descending=True)
# )

# Export

In [ ]:
(
  loandata_fe
  .write_excel(f'../result/bb_fe_account_{yyyymm}.xlsx')
)

In [ ]:
# (
#   loandata
#   .with_columns(
#     clean_iccode = pl.col('bot_isic_4_0').str.zfill(7).str.slice(1, 4)
#   )
#   .with_columns(
#     clean_iccode = (
#       pl.when(pl.col('grpbusi') == 'Individual')
#         .then(pl.lit('9609'))
#         .otherwise(pl.col('clean_iccode'))
#     )
#   )
#   .join(
#     ef_revenue_isic.drop('ic sector'),
#     how     = 'left', 
#     left_on  = 'clean_iccode',
#     right_on = 'clean_iccode',
#   )
#   .filter(
#     pl.col('scope_1_r').is_null()
#   )
#   .group_by('clean_iccode')
#   .len()
#   .sort('clean_iccode')
#   .write_clipboard()
# )